# Bank Marketing: synthetic smoke-эксперимент

Исполненный автономный эксперимент на **детерминированных синтетических smoke-данных**.
Он проверяет реальный код репозитория, но не оценивает качество на исходном публичном наборе.

## tl;dr

Ниже показан фактически исполненный smoke-run: объём синтетики, выбранная на validation
модель и метрики неизменяемого synthetic test split. Эти числа нельзя переносить на реальные данные.

In [1]:
import io
import json
import sys
import tempfile
from contextlib import redirect_stdout
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bank_marketing.data import TARGET, split_data, validate_frame
from bank_marketing.evaluate import main as evaluate_model
from bank_marketing.generate_smoke_data import generate_smoke_frame
from bank_marketing.train import main as train_model

SEED = 20250719
ROWS = 420
smoke_frame = generate_smoke_frame(rows=ROWS, seed=SEED)
validated_frame = validate_frame(smoke_frame)
train_frame, validation_frame, test_frame = split_data(validated_frame)

temporary_directory = tempfile.TemporaryDirectory(prefix="bank-marketing-smoke-")
run_directory = Path(temporary_directory.name)
data_path = run_directory / "smoke.csv"
artifact_path = run_directory / "model.joblib"
validation_path = run_directory / "validation.json"
metrics_path = run_directory / "test_metrics.json"
errors_path = run_directory / "test_errors.csv"
validated_frame.to_csv(data_path, index=False)

with redirect_stdout(io.StringIO()):
    train_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--report", str(validation_path), "--seed", str(SEED),
    ])
    evaluate_model([
        "--data", str(data_path), "--artifact", str(artifact_path),
        "--metrics", str(metrics_path), "--errors", str(errors_path),
    ])

validation_report = json.loads(validation_path.read_text(encoding="utf-8"))
test_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
error_rows = pd.read_csv(errors_path)
summary = pd.DataFrame([{
    "data": "deterministic synthetic smoke",
    "rows": len(validated_frame),
    "positive_rate": validated_frame[TARGET].mean(),
    "selected_model": test_metrics["model_name"],
    "test_pr_auc": test_metrics["pr_auc"],
    "test_roc_auc": test_metrics["roc_auc"],
    "test_brier": test_metrics["brier_score"],
    "test_recall_at_budget": test_metrics["recall_at_budget"],
}])
summary.round(4)

,data,rows,positive_rate,selected_model,test_pr_auc,test_roc_auc,test_brier,test_recall_at_budget
0,deterministic synthetic smoke,420,0.0381,gradient_boosting,0.0754,0.6063,0.1912,0.25


## Context & Methods


Цель — исполнить хронологический pre-contact workflow без доступа к UCI. Реальные функции обучения
и оценки читают временный CSV. Первые 60% строк служат train, следующие 20% validation,
последние 20% test; поле `duration` присутствует в smoke-таблице, но исключено из признаков как утечка.


            ### Key Assumptions


- Seed `20250719`, 420 упорядоченных строк; это синтетическая последовательность, не кампания банка.
- Исходный порядок строк считается приближением времени и не перемешивается.
- Бюджет обзвона 15% — учебный параметр; порог оценивается на validation.
- Brier score проверяет вероятности, но smoke-калибровка ничего не говорит о будущей кампании.

## Data

Smoke-таблица создаётся локальным генератором с фиксированным seed, затем проходит ту же
проверку схемы и то же разбиение, что и пользовательский CSV. Сетевые источники не используются.

In [2]:
display(pd.DataFrame({
    "split": ["train (past)", "validation", "test (future)"],
    "rows": [len(train_frame), len(validation_frame), len(test_frame)],
    "positive_rate": [
        train_frame[TARGET].mean(), validation_frame[TARGET].mean(), test_frame[TARGET].mean()
    ],
    "first_month": [train_frame["month"].iloc[0], validation_frame["month"].iloc[0], test_frame["month"].iloc[0]],
    "last_month": [train_frame["month"].iloc[-1], validation_frame["month"].iloc[-1], test_frame["month"].iloc[-1]],
}).round(4))
display(validated_frame.head(3))

,split,rows,positive_rate,first_month,last_month
0,train (past),252,0.0397,mar,aug
1,validation,84,0.0238,sep,oct
2,test (future),84,0.0476,nov,dec


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,duration,y
0,28,retired,divorced,university.degree,no,yes,unknown,cellular,mar,wed,...,999,0,nonexistent,1.472637,94.130796,-33.983561,4.870598,5215.326728,486,1
1,65,management,divorced,university.degree,no,no,no,telephone,mar,tue,...,999,0,nonexistent,1.458987,93.967030,-37.465506,4.810610,5222.013389,759,0
2,31,technician,single,high.school,no,no,no,cellular,mar,tue,...,999,0,failure,1.361991,93.988506,-35.164798,4.750439,5220.857442,838,0


## Results

Первая таблица — сравнение кандидатов на validation. Следующие результаты относятся только
к synthetic test split; таблица ошибок ограничена несколькими строками.

In [3]:
validation_table = pd.DataFrame(validation_report["models"]).T
display(validation_table[["pr_auc", "roc_auc", "brier_score", "precision", "recall", "f1"]]
        .sort_values("pr_auc", ascending=False).round(4))
display(pd.DataFrame({
    "metric": ["pr_auc", "roc_auc", "brier_score", "precision", "recall", "f1", "recall_at_budget"],
    "synthetic_test": [test_metrics[key] for key in
                       ["pr_auc", "roc_auc", "brier_score", "precision", "recall", "f1", "recall_at_budget"]],
}).round(4))

,pr_auc,roc_auc,brier_score,precision,recall,f1
gradient_boosting,0.082108,0.658537,0.11071,0.076923,0.5,0.133333
random_forest,0.040203,0.518293,0.112922,0.0,0.0,0.0
calibrated_random_forest,0.0386,0.5,0.023919,0.0,0.0,0.0
weighted_logistic_regression,0.026398,0.341463,0.21434,0.0,0.0,0.0
logistic_regression,0.024729,0.29878,0.02757,0.0,0.0,0.0
dummy,0.02381,0.5,0.023495,0.02381,1.0,0.046512


,metric,synthetic_test
0,pr_auc,0.0754
1,roc_auc,0.6063
2,brier_score,0.1912
3,precision,0.0556
4,recall,0.2500
5,f1,0.0909
6,recall_at_budget,0.2500


In [4]:
error_rows.head(8)

,job,contact,month,campaign,y,score,prediction,error_type
0,admin.,telephone,dec,1,0,0.990895,1,false_positive
1,admin.,cellular,nov,1,0,0.964800,1,false_positive
2,retired,telephone,dec,1,0,0.944062,1,false_positive
3,management,telephone,nov,1,0,0.944020,1,false_positive
4,management,cellular,nov,1,0,0.940100,1,false_positive
5,admin.,cellular,nov,1,0,0.935153,1,false_positive
6,retired,cellular,nov,1,0,0.920527,1,false_positive
7,technician,cellular,nov,1,0,0.917734,1,false_positive


## Takeaways

Выводы ниже сформированы из сохранённых outputs текущего запуска и относятся только к smoke-проверке.

In [5]:
print(f"- На synthetic future test выбран {test_metrics['model_name']}: "
      f"PR-AUC={test_metrics['pr_auc']:.4f}, Brier={test_metrics['brier_score']:.4f}.")
print(f"- Synthetic recall при validation-пороге бюджета: {test_metrics['recall_at_budget']:.4f}; "
      f"строк FP/FN: {len(error_rows)}.")
print("- Результат проверяет temporal pipeline и защиту от duration leakage, но не качество на UCI.")

- На synthetic future test выбран gradient_boosting: PR-AUC=0.0754, Brier=0.1912.
- Synthetic recall при validation-пороге бюджета: 0.2500; строк FP/FN: 20.
- Результат проверяет temporal pipeline и защиту от duration leakage, но не качество на UCI.
